In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN
from keras.optimizers import Adam

In [5]:
# Load the dataset
data = pd.read_csv('imbalance_data_CIC-IDS-2017 Dataset.csv')

In [6]:
# Display the first few rows of the dataset
print(data.head())

   Unnamed: 0  flow_duration  total_fwd_packets  total_backward_packets  \
0           0            3.0                2.0                     0.0   
1           1          109.0                1.0                     1.0   
2           2           52.0                1.0                     1.0   
3           3           34.0                1.0                     1.0   
4           4            3.0                2.0                     0.0   

   flow_bytes/s  flow_packets/s  packet_length_mean  packet_length_std  \
0  4.000000e+06    666666.66670                 6.0                0.0   
1  1.100917e+05     18348.62385                 6.0                0.0   
2  2.307692e+05     38461.53846                 6.0                0.0   
3  3.529412e+05     58823.52941                 6.0                0.0   
4  4.000000e+06    666666.66670                 6.0                0.0   

   flow_iat_mean  fwd_iat_mean  bwd_iat_mean  syn_flag_count  ack_flag_count  \
0            3.0        

In [7]:
# Separate features and labels
X = data.drop(columns=['label', 'Unnamed: 0'])
y = data['label']

In [8]:
# Handle infinite values and large values in the features
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.mean(), inplace=True)

In [9]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# Reshape the data for RNN input (samples, time steps, features)
X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

In [12]:
# Build the Error-Resilient Recurrent Neural Network model
model = Sequential()
model.add(SimpleRNN(50, input_shape=(1, X_train_scaled.shape[2]), return_sequences=True))
model.add(SimpleRNN(50))
model.add(Dense(1, activation='sigmoid'))

C:\Users\Dell\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [13]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

In [14]:

# Train the model
model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test))

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f'Test Loss: {loss}, Test Accuracy: {accuracy}')

Epoch 1/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 223s 3ms/step - accuracy: 0.5287 - loss: -588.0231 - val_accuracy: 0.5648 - val_loss: -2353.2742
Epoch 2/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 223s 3ms/step - accuracy: 0.5654 - loss: -2966.5525 - val_accuracy: 0.5782 - val_loss: -4743.8965
Epoch 3/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 230s 3ms/step - accuracy: 0.5678 - loss: -5359.1592 - val_accuracy: 0.5573 - val_loss: -7136.8691
Epoch 4/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 198s 3ms/step - accuracy: 0.5729 - loss: -7740.5850 - val_accuracy: 0.5785 - val_loss: -9542.8770
Epoch 5/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 201s 3ms/step - accuracy: 0.5750 - loss: -10215.8848 - val_accuracy: 0.5808 - val_loss: -12011.4570
Epoch 6/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 198s 3ms/step - accuracy: 0.5788 - loss: -12628.4199 - val_accuracy: 0.5807 - val_loss: -14436.4346
Epoch 7/10
70730/70730 ━━━━━━━━━━━━━━━━━━━━ 186s 3ms/step - accuracy: 0.5798 - loss: -15068.2979 - val_accuracy: 0.5842 - val_loss: -16842.9180
E